# Part 13 - Online Evals: an A/B test with a guardrail, by hand

*Evals from First Principles, Part 13.*

Every part so far has scored a model offline, on a fixed set of examples we already had the answers for. That is necessary, but it is not the whole story: an offline win only ever *predicts* a production win, it does not prove one. The only place a change proves itself is live traffic, under a controlled experiment. Here we run one by hand: a support chatbot ships a new answer-generation prompt and splits one week of traffic 50/50 between the old prompt (control) and the new one (treatment). We track a PRIMARY business metric - resolution rate - and a GUARDRAIL metric - refusal rate - that must not regress even if the primary metric wins. We derive the two-proportion z-test and its p-value by hand, apply a guardrail margin, and land on a ship / no-ship verdict.

Pure Python and stdlib math, offline, deterministic. The normal CDF is computed from `math.erf` (exact, no SciPy). A silent SciPy cross-check runs only if SciPy happens to be installed; it never changes what is printed.

In [ ]:
import math


## Step 1 - The experiment

We shipped a new answer-generation prompt to a support chatbot and split live traffic 50/50 for a week. Every session is one trial.

- **PRIMARY metric = resolution rate**: did the bot resolve the issue without escalating to a human? (higher is better)
- **GUARDRAIL metric = refusal rate**: did the bot wrongly say "I can't help"? (lower is better; a business guardrail, not an offline score - you only see it in production)

The counts are the tiny, traceable heart of the whole test: two arms, each with a session count, a resolved count, and a refused count. We also fix the significance level (`ALPHA`) and how much the guardrail is allowed to move before we call it a breach (`GUARDRAIL_MARGIN`).

In [ ]:
CONTROL   = dict(name="control  (old prompt)", n=400, resolved=168, refused=20)
TREATMENT = dict(name="treatment(new prompt)", n=400, resolved=200, refused=40)

ALPHA = 0.05            # significance level for the primary metric
GUARDRAIL_MARGIN = 0.02  # treatment refusal rate may rise at most +2 pp


## Step 2 - A proportion, by hand

Every rate in this notebook - resolution rate, refusal rate, pooled rate - is the same shape: successes divided by trials. One tiny helper covers all of them.

In [ ]:
def rate(count, n):
    """A conversion-style proportion: successes / trials."""
    return count / n if n else 0.0


## Step 3 - The normal CDF, by hand

The two-proportion z-test turns an observed lift into a z-score, then asks: "if there were truly no difference between the arms, how likely is a z-score at least this extreme?" That likelihood comes from the standard normal CDF, `Phi(z)`. We do not import SciPy for this - `math.erf` gives an exact closed form: `Phi(z) = 0.5 * (1 + erf(z / sqrt(2)))`.

In [ ]:
def normal_cdf(z):
    """Standard-normal CDF via the error function. Phi(z)=0.5(1+erf(z/sqrt2))."""
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


## Step 4 - The two-proportion z-test, by hand

Now we can ask whether the observed lift is real or noise. The steps:

1. `p1`, `p2` - the two observed rates.
2. `p_pool` - the rate the two arms would share if the prompt made no difference at all (all successes over all trials, pooled).
3. `se` - the standard error of `p2 - p1` under that pooled null: `sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))`.
4. `z` - how many standard errors the observed gap is from zero: `(p2 - p1) / se`.
5. `p_value` - two-sided: `2 * P(Z > |z|) = 2 * (1 - Phi(|z|))`.

The function returns all of these intermediate numbers so we can print each step, not just the final verdict.

In [ ]:
def two_proportion_ztest(x1, n1, x2, n2):
    """
    Two-proportion z-test, every step by hand.
      p1, p2   : the two observed rates
      p_pool   : rate if the arms were identical (all successes / all trials)
      se       : standard error of (p2 - p1) under that pooled null
      z        : (p2 - p1) / se
      p_value  : two-sided, 2 * P(Z > |z|)
    Returns a dict of the intermediate numbers so the caller can print them.
    """
    p1 = rate(x1, n1)
    p2 = rate(x2, n2)
    p_pool = (x1 + x2) / (n1 + n2)
    se = math.sqrt(p_pool * (1.0 - p_pool) * (1.0 / n1 + 1.0 / n2))
    z = (p2 - p1) / se
    p_value = 2.0 * (1.0 - normal_cdf(abs(z)))
    return dict(p1=p1, p2=p2, p_pool=p_pool, se=se, z=z, p_value=p_value)


## Step 5 - A tiny formatting helper

Just a horizontal rule for the printed report, same trick as earlier parts in the series.

In [ ]:
def bar(char="=", width=72):
    return char * width


## Step 6 - The two arms of the experiment

Print what we are working with before deriving anything: two arms, one week, 50/50 split.

In [ ]:
print(bar())
print("PART 13 - ONLINE EVALS: an A/B test with a guardrail, by hand.")
print(bar())

print("Two arms of live traffic, one week, 50/50 split:")
for arm in (CONTROL, TREATMENT):
    print(f"  {arm['name']}:  n={arm['n']}  "
          f"resolved={arm['resolved']}  refused={arm['refused']}")


## Step 7 - The primary metric: resolution rate

The business metric we actually care about. We compute the observed rate in each arm, the absolute lift (treatment minus control, in percentage points), and the relative lift (the absolute lift as a fraction of the control rate).

In [ ]:
p_c = rate(CONTROL["resolved"], CONTROL["n"])
p_t = rate(TREATMENT["resolved"], TREATMENT["n"])
abs_lift = p_t - p_c
rel_lift = abs_lift / p_c

print("\n" + bar("-"))
print("PRIMARY METRIC - resolution rate (higher is better)")
print(bar("-"))
print(f"  control   = {CONTROL['resolved']}/{CONTROL['n']} = {p_c:.3f}")
print(f"  treatment = {TREATMENT['resolved']}/{TREATMENT['n']} = {p_t:.3f}")
print(f"  absolute lift = {p_t:.3f} - {p_c:.3f} = {abs_lift:+.3f}  "
      f"({abs_lift*100:+.1f} pp)")
print(f"  relative lift = {abs_lift:.3f}/{p_c:.3f} = {rel_lift*100:+.1f}%")


## Step 8 - Is the lift real, or noise?

A +8 pp lift looks good, but with 400 sessions per arm it could still be luck. We run the two-proportion z-test derived above and compare the p-value against `ALPHA`.

In [ ]:
t = two_proportion_ztest(CONTROL["resolved"], CONTROL["n"],
                         TREATMENT["resolved"], TREATMENT["n"])
print("\n  Is it real, or noise? two-proportion z-test:")
print(f"    pooled rate p = ({CONTROL['resolved']}+{TREATMENT['resolved']})"
      f"/({CONTROL['n']}+{TREATMENT['n']}) = {t['p_pool']:.3f}")
print(f"    SE = sqrt(p(1-p)(1/n_c + 1/n_t)) = {t['se']:.5f}")
print(f"    z  = (p_t - p_c)/SE = {abs_lift:+.3f}/{t['se']:.5f} = {t['z']:.3f}")
print(f"    p-value (two-sided) = {t['p_value']:.4f}")
primary_sig = t["p_value"] < ALPHA
verdict = "SIGNIFICANT" if primary_sig else "not significant"
print(f"    at alpha={ALPHA}: {verdict}  (p {'<' if primary_sig else '>='} {ALPHA})")


## Step 9 - The guardrail metric: refusal rate

A win on the primary metric is not the whole story. The new prompt must also not make the bot refuse more often. We compute the same shape of numbers - observed rate per arm, the change in percentage points - and compare the change against the allowed margin.

In [ ]:
r_c = rate(CONTROL["refused"], CONTROL["n"])
r_t = rate(TREATMENT["refused"], TREATMENT["n"])
guard_delta = r_t - r_c
guard_ok = guard_delta <= GUARDRAIL_MARGIN

print("\n" + bar("-"))
print("GUARDRAIL METRIC - refusal rate (must NOT regress, lower is better)")
print(bar("-"))
print(f"  control   = {CONTROL['refused']}/{CONTROL['n']} = {r_c:.3f}")
print(f"  treatment = {TREATMENT['refused']}/{TREATMENT['n']} = {r_t:.3f}")
print(f"  change = {r_t:.3f} - {r_c:.3f} = {guard_delta:+.3f}  "
      f"({guard_delta*100:+.1f} pp)")
print(f"  allowed margin = +{GUARDRAIL_MARGIN*100:.0f} pp  ->  "
      f"{'OK' if guard_ok else 'BREACHED'}")


## Step 10 - The ship / no-ship verdict

Ship only if BOTH hold: the primary metric win is statistically significant, AND the guardrail did not breach its margin. Either condition failing alone is enough to hold the ship.

In [ ]:
ship = primary_sig and guard_ok
print("\n" + bar())
print("VERDICT: ship only if primary is SIGNIFICANT and guardrail is OK.")
print(f"  primary significant? {primary_sig}   guardrail ok? {guard_ok}")
if ship:
    print("  -> SHIP IT.")
else:
    print("  -> DO NOT SHIP. The new prompt resolves more issues (a real,")
    print("     significant +8 pp gain) but refuses twice as often, breaching")
    print("     the guardrail. An offline win is necessary, not sufficient.")
print(bar())


## Step 11 - A silent cross-check

If SciPy happens to be installed, we quietly confirm our hand-derived p-value agrees with `scipy.stats.norm`. This never prints anything and never changes the verdict - SciPy only verifies, it never replaces the derivation.

In [ ]:
try:
    from scipy.stats import norm
    assert abs(t["p_value"] - 2 * (1 - norm.cdf(abs(t["z"])))) < 1e-12
except ImportError:
    pass


## Recap

- An offline score only ever *predicts* a production outcome. The only place a change proves itself is live traffic, under a controlled experiment.
- A two-proportion z-test turns an observed lift (control resolution rate 0.420 vs treatment 0.500, a +8.0 pp / +19.0% lift) into a p-value (0.0232) so you can tell a real effect from noise, at a chosen significance level (`alpha=0.05`).
- A **guardrail metric** (here, refusal rate) is a metric that must not regress even when the primary metric wins. It gets its own margin (+2 pp here), separate from the primary metric's significance test.
- The ship / no-ship verdict needs BOTH conditions: primary significant AND guardrail intact. Here the primary metric wins significantly (p=0.0232 < 0.05) but the guardrail breaches (refusal rate doubles, +5.0 pp against a +2 pp allowance) - so the verdict is DO NOT SHIP.

Next: how do you keep re-running this kind of test safely, part after part, without an old regression sneaking back in unnoticed?